# plan-a-closure Tier 0 with the long trial, on a Kaggle GPU (engineering shakedown)

This runs the one-command Tier 0 pilot with its long trial (`closure tier0 --trial-profile long`) on the GPU this
Kaggle session has. The trial runs about 9 hours, the longest that fits Kaggle's 12-hour limit, which is what the
pre-registered decision rules need to be read at all: 204 null blocks for the false-alarm rule (H1.2 needs 200; the
one-hour smoke trial has about 30) and 12 hidden blocks for every class and intensity, with calibration blocks
interleaved through the run so slow drift reaches calibration and audit alike (amendment v0.5).

**What it is.** A shakedown on a non-target device, in a container, with NVML as the only power source.

**What it is not.** The registered trial. There is no independent power meter, so assumption A6 does not hold, and the
registered numbers come from Tier 1 on instrumented H100/H200 hardware (`docs/PREREGISTRATION.md`). Every run is
published whatever it shows (`docs/DEVIATIONS.md`).

**How to run it** (about 10.5 hours, unattended; nothing waits for you)

1. Your Kaggle account must be phone-verified (Settings > Phone verification): Kaggle needs it for GPUs and internet.
2. In the notebook's right-hand panel: Accelerator **GPU T4 x2** (not P100: it has no NVML energy counter) and
   Internet **on**. Check that at least 11 GPU hours of this week's quota are left.
3. **Save Version** (top right) > **Save & Run All (Commit)** > Save. The committed run continues on its own after
   you close the tab. Then stop the interactive editing session (its power icon), so it does not use quota as well.
4. After about 10.5 hours, open the saved version (the notebook's Versions list) > **Output** and download
   `tier0_bundle.tar.gz`, `SUMMARY.md` and `drill.json`.

The private signing key never reaches the saved output: the run happens outside `/kaggle/working`, and only those three
files are copied there. Keep the notebook private until you publish the result.

In [ ]:
# 1. Setup: the GPU, internet, the exact code. The run happens under /tmp, outside /kaggle/working (which Kaggle saves
#    with the version), so only the three deliverables are saved, never the private signing key.
import os, shutil, subprocess, time
REPO, REF = "https://github.com/AnonRish/plan-a-closure", "main"   # REF can be a tag or a commit, to pin the code
SRC, RUNS, SAVE = "/tmp/plan-a-closure", "/tmp/plan-a-closure-runs", "/kaggle/working"
T_START = time.time()
q = subprocess.run(["nvidia-smi", "--query-gpu=name,uuid,driver_version,memory.total,power.limit", "--format=csv"],
                   capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if q is None or q.returncode != 0 or "\n" not in q.stdout.strip():
    raise SystemExit("No GPU: in the right-hand panel set Accelerator to GPU T4 x2, then Save & Run All again.")
print(q.stdout)
if "P100" in q.stdout:
    raise SystemExit("This is a P100, which has no NVML energy counter: choose GPU T4 x2, then Save & Run All again.")
try:
    online = subprocess.run(["git", "ls-remote", "-q", REPO, "HEAD"], capture_output=True, timeout=60).returncode == 0
except subprocess.TimeoutExpired:
    online = False
if not online:
    raise SystemExit("No internet: turn Internet on in the right-hand panel (it needs a phone-verified account).")
if not os.path.isdir(f"{SRC}/.git"):
    !git clone -q --depth 1 {REPO} {SRC}
!git -C {SRC} fetch -q --depth 1 origin {REF} && git -C {SRC} checkout -q FETCH_HEAD
%cd /tmp/plan-a-closure
!pip install -q -e ".[test,hw]"
!python -c "import closure; print('plan-a-closure', closure.__version__)"
!git log -1 --format="code: commit %H (%cd)"
!nvidia-smi -pm 1 > /dev/null 2>&1 && echo "persistence mode on" || echo "persistence mode not available here (fine)"

In [ ]:
# 2. Preflight: NVML, the energy counter, torch and CUDA, which physical GPU. FAIL lines here stop the run in cell 3;
#    WARN lines for fp8, clock sync and tcpdump are normal on Kaggle.
!closure preflight --gpu 0

## The run (about 10 hours)

Cell 3 runs Tier 0 with the long trial, then A Narrow Path's mock training run on the same GPU, then copies the three
deliverables into `/kaggle/working`, all in one cell. Tier 0 logs every step, sizes the unit of work to this GPU,
locks the trial's clocks where the container allows it (the log says whether it did), restores the GPU on exit, and
writes `SUMMARY.md` and `tier0_bundle.tar.gz`. If the run is still going 11 hours 20 minutes after the session started,
it is stopped cleanly so the summary and the bundle of every finished step are still saved before Kaggle's 12-hour
limit.

In [ ]:
# 3. The run: Tier 0 with the long trial, the mock training run, then the deliverables into /kaggle/working.
import hashlib
from IPython.display import Markdown, display
OUT = time.strftime(f"{RUNS}/tier0-%Y%m%d-%H%M%S")                  # a fresh folder for every run
LEFT = int(11 * 3600 + 20 * 60 - (time.time() - T_START))           # stop cleanly before Kaggle's 12-hour limit
!timeout --signal=TERM {LEFT} closure tier0 --confirm-hardware --gpu 0 --trial-profile long --out {OUT}
if os.path.exists(f"{OUT}/probe.json") and time.time() - T_START < 11.5 * 3600:
    !closure treaty mock-run --device cuda:0 --probe {OUT}/probe.json --budget-flop 1e16 --dim 4096 --layers 8 --batch 2048 --out {OUT}/mock-run
keep = [p for p in (f"{OUT}/tier0_bundle.tar.gz", f"{OUT}/SUMMARY.md", f"{OUT}/mock-run/drill.json") if os.path.exists(p)]
dest = os.path.join(SAVE, os.path.basename(OUT))
os.makedirs(dest, exist_ok=True)
for p in keep:
    shutil.copy2(p, dest)
print("saved with this version's output:", dest, sorted(os.listdir(dest)))
if os.path.exists(f"{OUT}/tier0_bundle.tar.gz"):
    print("bundle sha256:", hashlib.sha256(open(f"{OUT}/tier0_bundle.tar.gz", "rb").read()).hexdigest())
if os.path.exists(f"{OUT}/SUMMARY.md"):
    display(Markdown(open(f"{OUT}/SUMMARY.md", encoding="utf-8").read()))

## After the run

* Download `tier0_bundle.tar.gz`, `SUMMARY.md` and `drill.json` from the saved version's Output. On Windows,
  `Get-FileHash $HOME\Downloads\tier0_bundle.tar.gz` should match the sha256 printed above.
* `SUMMARY.md` has a section that reads the pre-registered decision rules at this run's scale (`closure criteria`):
  which of H1.1 to H1.4 the data decide, and how. It is a shakedown reading: NVML only, so no Plan A claim.
* Publish the result whatever it shows, labelled as a shakedown: a new section in `docs/DEVIATIONS.md` (date, GPU,
  "Kaggle container", no independent meter, the bundle's SHA-256) and the bundle under `results/`.
* The next step is the registered run: Tier 0 and then Tier 1 on instrumented H100/H200 hardware
  (`docs/HARDWARE_CONFIG.md`).